1. Setup


In [2]:
from google.cloud import bigquery
from google import genai
from google.genai import types

PROJECT_ID = "qwiklabs-gcp-04-c5701ae3d55f"
LOCATION = "us-central1"
DATASET = "AuroraBay"

bq = bigquery.Client(project=PROJECT_ID)
client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)

2. Create dataset + load the CSV. print out some sample rows for confirmation:


In [10]:
from google.cloud import bigquery

# Use the active project so the notebook is portable across lab environments
bq = bigquery.Client()
PROJECT_ID = bq.project
DATASET = "AuroraBay"
TABLE = f"{PROJECT_ID}.{DATASET}.faqs"
SOURCE_URI = "gs://labs.roitraining.com/aurora-bay-faqs/aurora-bay-faqs.csv"

# Ensure the dataset exists (US, to match the embedding connection later)
bq.create_dataset(bigquery.Dataset(f"{PROJECT_ID}.{DATASET}"), exists_ok=True)

# Load the CSV with an explicit schema; skip the header, overwrite on re-run
load_job = bq.load_table_from_uri(
    SOURCE_URI,
    TABLE,
    job_config=bigquery.LoadJobConfig(
        source_format=bigquery.SourceFormat.CSV,
        skip_leading_rows=1,
        write_disposition="WRITE_TRUNCATE",
        schema=[
            bigquery.SchemaField("question", "STRING"),
            bigquery.SchemaField("answer", "STRING"),
        ],
    ),
)
load_job.result()  # block until the load completes

# Confirm the load succeeded — row count + a sample to confirm
table = bq.get_table(TABLE)
print(f"Loaded {table.num_rows} rows into {TABLE}")
for row in bq.list_rows(table, max_results=3):
    print(dict(row))

Loaded 50 rows into qwiklabs-gcp-04-c5701ae3d55f.AuroraBay.faqs
{'question': 'When was Aurora Bay founded?', 'answer': 'Aurora Bay was founded in 1901 by a group of fur traders who recognized the region’s strategic coastal location.'}
{'question': 'What is the population of Aurora Bay?', 'answer': 'Aurora Bay has a population of approximately 3,200 residents, although it can fluctuate seasonally due to temporary fishing and tourism workforces.'}
{'question': 'Where is the Aurora Bay Town Hall located?', 'answer': 'The Town Hall is located at 100 Harbor View Road, in the center of Aurora Bay, close to the main harbor.'}


Getting the column names:

In [11]:
print([f.name for f in bq.get_table(f"{PROJECT_ID}.{DATASET}.faqs").schema])

['question', 'answer']


Creating the big query connection, as well as IAM:

In [14]:
!bq mk --connection --location=US --connection_type=CLOUD_RESOURCE embedding_conn

Connection 560612806950.us.embedding_conn successfully created


In [17]:
import json, subprocess

conn = json.loads(subprocess.check_output(
    ["bq", "show", "--format=json", "--connection", "us.embedding_conn"]))
sa = conn["cloudResource"]["serviceAccountId"]
print("Service account:", sa)

project = bq.project
subprocess.run([
    "gcloud", "projects", "add-iam-policy-binding", project,
    f"--member=serviceAccount:{sa}",
    "--role=roles/aiplatform.user",
], check=True)

Service account: bqcx-560612806950-bws3@gcp-sa-bigquery-condel.iam.gserviceaccount.com


CompletedProcess(args=['gcloud', 'projects', 'add-iam-policy-binding', 'qwiklabs-gcp-04-c5701ae3d55f', '--member=serviceAccount:bqcx-560612806950-bws3@gcp-sa-bigquery-condel.iam.gserviceaccount.com', '--role=roles/aiplatform.user'], returncode=0)

Checking the big query connection

In [18]:
!bq ls --connection --location=US

               name                friendlyName   description    Last modified         type        hasCredential                                            properties                                            
 -------------------------------- -------------- ------------- ----------------- ---------------- --------------- ----------------------------------------------------------------------------------------------- 
  560612806950.us.embedding_conn                                15 Jun 17:11:50   CLOUD_RESOURCE   False           {"serviceAccountId": "bqcx-560612806950-bws3@gcp-sa-bigquery-condel.iam.gserviceaccount.com"}  


Model Creation:

In [19]:
CONNECTION = "us.embedding_conn"
EMBED_MODEL = f"{DATASET}.Embeddings"

bq.query(f"""
CREATE OR REPLACE MODEL `{EMBED_MODEL}`
REMOTE WITH CONNECTION `{CONNECTION}`
OPTIONS (ENDPOINT = 'text-embedding-005')
""").result()

print(f"Created remote model {EMBED_MODEL}")

Created remote model AuroraBay.Embeddings


Generate embeddings for each Q&A pair and store them:

In [20]:
EMBED_TABLE = f"{DATASET}.faqs_embedded"

bq.query(f"""
CREATE OR REPLACE TABLE `{EMBED_TABLE}` AS
SELECT *
FROM ML.GENERATE_EMBEDDING(
  MODEL `{EMBED_MODEL}`,
  (SELECT
     question,
     answer,
     CONCAT(question, ' ', answer) AS content
   FROM `{DATASET}.faqs`)
)
""").result()

# Confirm: row count + check the vector length
table = bq.get_table(EMBED_TABLE)
print(f"Embedded {table.num_rows} rows into {EMBED_TABLE}")

row = next(iter(bq.list_rows(table, max_results=1)))
print("Embedding dimensions:", len(row["ml_generate_embedding_result"]))

Embedded 50 rows into AuroraBay.faqs_embedded
Embedding dimensions: 768


Vector search retrieval:


In [23]:
def search_faqs(user_q: str, top_k: int = 3):
    """Return the top_k most relevant FAQ rows for a user question."""
    sql = f"""
    SELECT base.question, base.answer, distance
    FROM VECTOR_SEARCH(
      TABLE `{EMBED_TABLE}`,
      'ml_generate_embedding_result',
      (SELECT ml_generate_embedding_result
       FROM ML.GENERATE_EMBEDDING(
         MODEL `{EMBED_MODEL}`,
         (SELECT @q AS content))),
      top_k => {top_k},
      options => '{{"fraction_lists_to_search": 1.0}}')
    ORDER BY distance
    """
    job = bq.query(sql, job_config=bigquery.QueryJobConfig(
        query_parameters=[bigquery.ScalarQueryParameter("q", "STRING", user_q)]
    ))
    return [dict(r) for r in job.result()]


Generating results in Gemini

In [25]:
from google import genai

client = genai.Client(vertexai=True, project=bq.project, location="us-central1")
GEN_MODEL = "gemini-2.5-flash"

SYSTEM_INSTRUCTION = (
    "You are Chaleb, a helpful assistant for the town of Aurora Bay, Alaska. "
    "Answer the user's question using ONLY the provided context. "
    "If the context does not contain the answer, say you don't have that "
    "information rather than guessing."
)

def answer_question(user_q: str, top_k: int = 3) -> str:
    """Full RAG: retrieve relevant FAQs, then generate a grounded answer."""
    hits = search_faqs(user_q, top_k=top_k)
    context = "\n\n".join(
        f"Q: {h['question']}\nA: {h['answer']}" for h in hits
    )

    resp = client.models.generate_content(
        model=GEN_MODEL,
        contents=f"Context:\n{context}\n\nQuestion: {user_q}",
        config=genai.types.GenerateContentConfig(
            system_instruction=SYSTEM_INSTRUCTION,
            temperature=0.2,
        ),
    )
    return resp.text

# End-to-end test
print(answer_question("Where is the town hall located?"))
print("---")
print(answer_question("What's in Caleb's Chill Branch?"))  # out-of-scope check

The Town Hall is located at 100 Harbor View Road, in the center of Aurora Bay, close to the main harbor.
---
I don't have that information.
